<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/conversational_autogen_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Uninstall any conflicting versions first
!pip uninstall -y autogen autogen-agentchat pyautogen -q

# Install the correct package
!pip install autogen -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 9.8 MB/s eta 0:00:00


In [ ]:
import autogen
print(autogen.__version__)

0.11.4


In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://openai.vocareum.com/v1'

# AutoGen also needs the base_url in its config_list.
# We store it here so all agents below can reference it.
OPENAI_BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://openai.vocareum.com/v1')
OPENAI_API_KEY  = os.environ.get('OPENAI_API_KEY')



## ConversableAgent — Core Building Block

A `ConversableAgent` wraps an LLM and defines:
- **`system_message`** — the persona / role instructions given to the model.
- **`llm_config`** — which model to use and how to reach it (API key, model name, temperature).
- **`human_input_mode`** — one of `NEVER`, `ALWAYS`, or `TERMINATE`.
  - `NEVER`: agent never asks for human input (fully autonomous).
  - `ALWAYS`: pauses and waits for a human reply after every message.
  - `TERMINATE`: asks for human input only when the auto-reply limit is reached or a termination condition fires.
- **`code_execution_config`** — whether the agent can execute code in a local sandbox.


In [ ]:
from autogen import ConversableAgent
# Create a single autonomous chatbot agent.
# - 'chatbot' is the agent's display name used in conversation logs.
# - llm_config specifies the model and API key; config_list allows multiple fallback models.

# - human_input_mode='NEVER' means the agent never pauses to ask a real human.

agent = ConversableAgent(
    name = "chatbot",
    llm_config={"config_list": [{"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}]},
    human_input_mode="NEVER")

In [ ]:
agent.generate_reply(messages=[{"content": "Tell me a joke" , "role" : "user"}])

'Why did the scarecrow win an award? \n\nBecause he was outstanding in his field!'

In [ ]:
# Multi Agents

# We will create two diffrent agents and make them talk to each other

peter = ConversableAgent(
    name = 'peter',
    system_message = 'Your name is Peter. And you are a duo comedian . You have to crack jokes',
     llm_config={"config_list": [{"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}]},
     human_input_mode="NEVER")


In [ ]:


# We will create two diffrent agents and make them talk to each other

mary = ConversableAgent(
    name = 'mary',
    system_message = 'Your name is mary. And you are a duo comedian . You have to crack jokes',
     llm_config={"config_list": [{"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}]},
     human_input_mode="NEVER")


In [ ]:
peter.initiate_chat(mary, message="Tell me a joke" , max_turns=2)

peter (to mary):

Tell me a joke

--------------------------------------------------------------------------------
mary (to peter):

Why don’t skeletons fight each other? 

They don’t have the guts!

--------------------------------------------------------------------------------
peter (to mary):

That's a classic, Mary! Here’s one for you: 

Why did the scarecrow win an award? 

Because he was outstanding in his field!

--------------------------------------------------------------------------------
mary (to peter):

Oh, that’s a good one, Peter! Here’s my comeback: 

Why did the bicycle fall over? 

Because it was two-tired!

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (5484f3d4-922f-4844-913e-a9b62e013a34): Maximum turns (2) reached


ChatResult(chat_id=258786375545056017340525991234323213766, chat_history=[{'content': 'Tell me a joke', 'role': 'assistant', 'name': 'peter'}, {'content': 'Why don’t skeletons fight each other? \n\nThey don’t have the guts!', 'role': 'user', 'name': 'mary'}, {'content': "That's a classic, Mary! Here’s one for you: \n\nWhy did the scarecrow win an award? \n\nBecause he was outstanding in his field!", 'role': 'assistant', 'name': 'peter'}, {'content': 'Oh, that’s a good one, Peter! Here’s my comeback: \n\nWhy did the bicycle fall over? \n\nBecause it was two-tired!', 'role': 'user', 'name': 'mary'}], summary='Oh, that’s a good one, Peter! Here’s my comeback: \n\nWhy did the bicycle fall over? \n\nBecause it was two-tired!', cost={'usage_including_cached_inference': {'total_cost': 7.514999999999998e-05, 'gpt-4o-mini-2024-07-18': {'cost': 7.514999999999998e-05, 'prompt_tokens': 185, 'completion_tokens': 79, 'total_tokens': 264}}, 'usage_excluding_cached_inference': {'total_cost': 7.5149999

## Human-in-the-Loop: Controlled Agent Interaction

A key strength of AutoGen is the ability to **inject human judgement** at the right moment:
- `max_consecutive_auto_reply=N` — after N automatic replies, AutoGen pauses and asks the human for input.
- `is_termination_msg=lambda msg: ...` — a custom function that inspects each message and stops the chat when a condition is met (e.g., the correct number is guessed).
- `human_input_mode='TERMINATE'` — human is consulted only at termination boundaries, not on every turn.

This pattern is essential for **human-oversight of agentic workflows** — the agent can run autonomously most of the time,
but a human can review, redirect, or approve at critical checkpoints.

In [ ]:
# Agent holding the secret number.
# max_consecutive_auto_reply=1 → after one auto-reply, AutoGen pauses for human input.
# is_termination_msg checks if '53' appears in the message — ends the game when guessed correctly.
# human_input_mode='TERMINATE' → human is consulted only at termination boundaries.


In [ ]:
agent_with_a_number = ConversableAgent(
    name = "agent_with_a_number",
    system_message= '''You are playing a game called guess my number ".
    In the first game, you have the
    number 53 in your mind, and I will try to guess it.
    If I guess too high, say 'too high', if I guess too low, say 'too low'. ''',
    llm_config={"config_list": [{"model": "gpt-4o-mini", "api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}]},
    max_consecutive_auto_reply=1,  # maximum number of consecutive auto-replies before asking for human input
    is_termination_msg=lambda msg: "53" in msg["content"],  # terminate if the number is guessed by the other agent
    human_input_mode="TERMINATE"  # ask for human input at termination boundaries
)


# The guessing agent — purely autonomous, never asks for human input.
agent_guess_number = ConversableAgent(
    "agent_guess_number",
    system_message="I have a number in my mind, and you will try to guess it. "
    "If I say 'too high', you should guess a lower number. If I say 'too low', "
    "you should guess a higher number. ",
    llm_config={"config_list": [{"model": "gpt-4", "api_key": OPENAI_API_KEY, "base_url": OPENAI_BASE_URL}]},
    human_input_mode="NEVER",
)


In [ ]:
agent_with_a_number.initiate_chat(agent_guess_number, message="I have a number between 1 and 100 . Can you guess it?" )

agent_with_a_number (to agent_guess_number):

I have a number between 1 and 100 . Can you guess it?

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_a_number):

Is it 50?

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
agent_with_a_number (to agent_guess_number):

Too low.

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_a_number):

Is it 75?

--------------------------------------------------------------------------------
Please give feedback to agent_guess_number. Press enter to skip and use auto-reply, or type 'exit' to stop the conversation: Too high
agent_with_a_number (to agent_guess_number):

Too high

--------------------------------------------------------------------------------
agent_guess_number (to agent_with_a_number):

Is it 62?

-----------------------------------------------

ChatResult(chat_id=149766751138860274695126873160818864968, chat_history=[{'content': 'I have a number between 1 and 100 . Can you guess it?', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Is it 50?', 'role': 'user', 'name': 'agent_guess_number'}, {'content': 'Too low.', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Is it 75?', 'role': 'user', 'name': 'agent_guess_number'}, {'content': 'Too high', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Is it 62?', 'role': 'user', 'name': 'agent_guess_number'}, {'content': 'Too high.', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Is it 56?', 'role': 'user', 'name': 'agent_guess_number'}, {'content': 'too high', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Is it 53?', 'role': 'user', 'name': 'agent_guess_number'}, {'content': 'yes', 'role': 'assistant', 'name': 'agent_with_a_number'}, {'content': 'Great! The number you had in mind was 53.', 'role'